# Phase 2: Baseline and GLMs

Build a flat-price baseline, then a Poisson GLM (frequency) and a Gamma GLM (severity), and check whether they beat the baseline on a validation set. The test set is not used in this notebook.

## Step 1: Load the training data

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, KBinsDiscretizer, FunctionTransformer, StandardScaler
from sklearn.linear_model import PoissonRegressor, GammaRegressor
from sklearn.metrics import mean_poisson_deviance, mean_gamma_deviance

train = pd.read_parquet("../data/train.parquet")
print(train.shape)
train.head()

(542410, 14)


,IDpol,ClaimNb,Exposure,Area,VehPower,VehAge,DrivAge,BonusMalus,VehBrand,VehGas,Density,Region,ClaimAmount,ClaimNbSev
0,3099378,1,1.00,B,6,5,59,50,B2,Diesel,96,R72,1204.0,1
1,3068220,0,1.00,C,10,11,61,50,B3,Regular,111,R91,0.0,0
2,2107013,0,1.00,A,7,4,42,50,B1,Diesel,19,R72,0.0,0
3,2121304,0,1.00,A,6,4,52,50,B2,Diesel,16,R82,0.0,0
4,6084903,0,0.28,D,5,3,71,50,B12,Regular,538,R53,0.0,0


## Step 2: Validation split

Models learn from `fit` and are scored on `val`.

In [2]:
fit, val = train_test_split(train, test_size=0.2, random_state=42)
fit, val = fit.copy(), val.copy()

print("Fit:", fit.shape)
print("Val:", val.shape)
print("Fit claim rate:", fit["ClaimNb"].sum() / fit["Exposure"].sum())
print("Val claim rate:", val["ClaimNb"].sum() / val["Exposure"].sum())

Fit: (433928, 14)
Val: (108482, 14)
Fit claim rate: 0.07291802976897951
Val claim rate: 0.07460470818221582


## BonusMalus cap

Extreme BonusMalus values (up to 230) made the GLM's predictions blow up exponentially. Capped at 150, which affects about 180 policies (0.03%). **This is a fixed rule: apply it to the test set too in Phase 4.**

In [3]:
print("Policies with BonusMalus > 150:", (train["BonusMalus"] > 150).sum())

for d in (fit, val):
    d["BonusMalus"] = d["BonusMalus"].clip(upper=150)

print("Max BonusMalus after cap:", fit["BonusMalus"].max(), val["BonusMalus"].max())

Policies with BonusMalus > 150: 180
Max BonusMalus after cap: 150 150


## Step 3: Claim cap

In [4]:
claims_fit = fit[fit["ClaimAmount"] > 0]
total = claims_fit["ClaimAmount"].sum()

for q in [0.99, 0.995, 0.999]:
    c = claims_fit["ClaimAmount"].quantile(q)
    excess = (claims_fit["ClaimAmount"] - c).clip(lower=0).sum()
    print(f"{q:.1%} cap at €{c:,.0f}: removes {excess / total:.1%} of cost")

99.0% cap at €17,924: removes 34.7% of cost
99.5% cap at €36,312: removes 29.5% of cost
99.9% cap at €183,748: removes 17.5% of cost


In [5]:
cap = claims_fit["ClaimAmount"].quantile(0.999)
print(f"Cap: €{cap:,.0f}")

for d in (fit, val):
    avg = d["ClaimAmount"] / d["ClaimNbSev"].replace(0, np.nan)
    d["ClaimAmountCapped"] = (avg.clip(upper=cap) * d["ClaimNbSev"]).fillna(0)

removed = fit["ClaimAmount"].sum() - fit["ClaimAmountCapped"].sum()
large_loss_load = removed / fit["Exposure"].sum()

print(f"Cost removed: {removed / fit['ClaimAmount'].sum():.1%}")
print(f"Large-loss surcharge: €{large_loss_load:,.2f} per policy-year")

Cap: €183,748
Cost removed: 16.5%
Large-loss surcharge: €29.15 per policy-year


## Claim cap

- Capped average claim cost at the 99.9th percentile of `fit` (about €184k); only 16 claims affected
- Lower caps (99%, 99.5%) would remove 30–35% of total cost, turning too much of the price into a flat surcharge
- Cap removed 16.5% of cost, added back as a flat large-loss surcharge of about €29 per policy-year

## Step 4: Prepare the features

Ages are split into 10 bands (risk by age is U-shaped), categories are one-hot encoded, and Density is logged. The preprocessor learns from `fit` only.

In [6]:
preprocessor = ColumnTransformer([
    ("age_bands", KBinsDiscretizer(n_bins=10, subsample=None), ["VehAge", "DrivAge"]),
    ("categories", OneHotEncoder(handle_unknown="ignore"),
        ["VehBrand", "VehPower", "VehGas", "Region", "Area"]),
    ("bonus", StandardScaler(), ["BonusMalus"]),
    ("log_density", make_pipeline(FunctionTransformer(np.log), StandardScaler()), ["Density"]),
])

preprocessor.fit(fit)
X_fit = preprocessor.transform(fit)
X_val = preprocessor.transform(val)

print("X_fit:", X_fit.shape)
print("X_val:", X_val.shape)

X_fit: (433928, 75)
X_val: (108482, 75)


## Step 5: Flat-price baseline

In [7]:
results = {}

# Frequency: claims per year
y_freq_val = val["ClaimNb"] / val["Exposure"]
base_rate = fit["ClaimNb"].sum() / fit["Exposure"].sum()
results["freq_baseline"] = mean_poisson_deviance(
    y_freq_val, np.full(len(val), base_rate), sample_weight=val["Exposure"]
)

# Severity: average capped cost per claim (policies with claims only)
sev_fit = fit[fit["ClaimNbSev"] > 0]
sev_val = val[val["ClaimNbSev"] > 0]
y_sev_val = sev_val["ClaimAmountCapped"] / sev_val["ClaimNbSev"]
base_sev = sev_fit["ClaimAmountCapped"].sum() / sev_fit["ClaimNbSev"].sum()
results["sev_baseline"] = mean_gamma_deviance(
    y_sev_val, np.full(len(sev_val), base_sev), sample_weight=sev_val["ClaimNbSev"]
)

flat_premium = base_rate * base_sev + large_loss_load

print(f"Baseline claim rate: {base_rate:.4f} claims per year")
print(f"Baseline cost per claim: €{base_sev:,.0f}")
print(f"Flat premium: €{flat_premium:,.2f} per year")

Baseline claim rate: 0.0729 claims per year
Baseline cost per claim: €2,026
Flat premium: €176.88 per year


## Step 6: Frequency GLM (Poisson)

In [8]:
y_freq_fit = fit["ClaimNb"] / fit["Exposure"]

glm_freq = PoissonRegressor(alpha=1e-4, max_iter=1000)
glm_freq.fit(X_fit, y_freq_fit, sample_weight=fit["Exposure"])
pred_freq_val = glm_freq.predict(X_val)

results["freq_glm"] = mean_poisson_deviance(
    y_freq_val, pred_freq_val, sample_weight=val["Exposure"]
)

print(f"Baseline deviance: {results['freq_baseline']:.4f}")
print(f"GLM deviance:      {results['freq_glm']:.4f}")
print(f"Improvement:       {1 - results['freq_glm'] / results['freq_baseline']:.1%}")
print("Predicted claims:", round((pred_freq_val * val["Exposure"]).sum()))
print("Actual claims:   ", val["ClaimNb"].sum())

Baseline deviance: 0.4818
GLM deviance:      0.4604
Improvement:       4.4%
Predicted claims: 4196
Actual claims:    4279


## Step 7: Severity GLM (Gamma)

Trained only on policies with claims. Try several regularization strengths and keep the best on validation.

In [9]:
X_sev_fit = preprocessor.transform(sev_fit)
X_sev_val = preprocessor.transform(sev_val)
y_sev_fit = sev_fit["ClaimAmountCapped"] / sev_fit["ClaimNbSev"]

for a in [1e-4, 1e-3, 1e-2, 0.1, 1, 10, 100]:
    m = GammaRegressor(alpha=a, max_iter=1000)
    m.fit(X_sev_fit, y_sev_fit, sample_weight=sev_fit["ClaimNbSev"])
    dev = mean_gamma_deviance(y_sev_val, m.predict(X_sev_val), sample_weight=sev_val["ClaimNbSev"])
    print(f"alpha={a:<7} deviance={dev:.4f}  improvement={1 - dev / results['sev_baseline']:.1%}")

alpha=0.0001  deviance=1.2181  improvement=-2.0%
alpha=0.001   deviance=1.2160  improvement=-1.9%
alpha=0.01    deviance=1.2079  improvement=-1.2%
alpha=0.1     deviance=1.1904  improvement=0.3%
alpha=1       deviance=1.1873  improvement=0.5%
alpha=10      deviance=1.1916  improvement=0.2%
alpha=100     deviance=1.1935  improvement=0.0%


In [10]:
glm_sev = GammaRegressor(alpha=1, max_iter=1000)
glm_sev.fit(X_sev_fit, y_sev_fit, sample_weight=sev_fit["ClaimNbSev"])

results["sev_glm"] = mean_gamma_deviance(
    y_sev_val, glm_sev.predict(X_sev_val), sample_weight=sev_val["ClaimNbSev"]
)

print(f"Baseline deviance: {results['sev_baseline']:.4f}")
print(f"GLM deviance:      {results['sev_glm']:.4f}")
print(f"Improvement:       {1 - results['sev_glm'] / results['sev_baseline']:.1%}")

Baseline deviance: 1.1938
GLM deviance:      1.1873
Improvement:       0.5%


## Severity findings

- With little regularization, the Gamma GLM did worse than the baseline (overfitting: about 20k noisy claims, 75 features)
- Stronger regularization helped; alpha = 1 was best, with only a small gain over the flat average
- Claim cost is hard to predict from driver and car features, consistent with the many fixed settlement amounts found in Phase 1

## Step 8: Combine into a premium

In [11]:
pred_sev_all_val = glm_sev.predict(X_val)
premium_val = pred_freq_val * pred_sev_all_val + large_loss_load

print(pd.Series(premium_val).describe())
print(f"Flat premium for comparison: €{flat_premium:,.2f}")

count    108482.000000
mean        185.751651
std         119.809412
min          67.358727
25%         126.466268
50%         150.615314
75%         195.894828
max        3780.176500
dtype: float64
Flat premium for comparison: €176.88


In [12]:
# Check the most expensive predictions
val_check = val.copy()
val_check["premium"] = premium_val
val_check["pred_freq"] = pred_freq_val
val_check["pred_sev"] = pred_sev_all_val

cols = ["premium", "pred_freq", "pred_sev", "DrivAge", "BonusMalus",
        "VehPower", "VehAge", "Density", "ClaimNb"]
val_check.sort_values("premium", ascending=False)[cols].head(10)

,premium,pred_freq,pred_sev,DrivAge,BonusMalus,VehPower,VehAge,Density,ClaimNb
108785,3780.176500,1.595460,2351.061177,75,150,6,4,5723,0
104435,3099.357046,1.329024,2310.120362,54,150,7,6,4116,0
204826,3001.075067,1.234878,2406.652907,46,148,4,1,731,0
100127,2926.028632,1.245659,2325.578247,53,150,7,5,4116,1
378466,2873.627676,1.176146,2418.472005,50,148,4,5,731,0
516434,2853.055407,1.143526,2469.471044,79,150,4,3,1182,1
433429,2837.703296,1.133697,2477.339502,80,150,4,4,1182,0
72173,2766.842247,1.148374,2383.971356,20,150,6,5,4762,2
175642,2710.330778,1.145247,2341.135205,52,150,6,8,269,0
449945,2680.663124,1.099631,2411.273196,48,150,6,10,141,0


In [13]:
print("All results:")
for k, v in results.items():
    print(f"  {k:<15} {v:.4f}")

All results:
  freq_baseline   0.4818
  sev_baseline    1.1938
  freq_glm        0.4604
  sev_glm         1.1873


## Phase 2 summary

| Model | Validation deviance | vs. baseline |
| --- | --- | --- |
| Frequency baseline | 0.4818 | — |
| Frequency GLM (Poisson) | 0.4604 | 4.4% better |
| Severity baseline | 1.1938 | — |
| Severity GLM (Gamma, alpha = 1) | 1.1873 | 0.5% better |

- Flat premium: €176.88 for every driver
- GLM premiums: median about €151, middle 50% between about €126 and €196, range about €67 to €3,780
- Most drivers pay less than the flat price; a small high-risk group pays much more
- Top predicted premiums all have BonusMalus near the 150 cap; calibration in Phase 4 will check whether they are overpriced